# AHP-MAUT Framework: Interactive Walkthrough

This notebook mirrors `run_all.py`, broken into explorable steps.

**Paper:** Silva et al. *Integrated AHP-MAUT Framework for Multicriteria Performance Evaluation of Soybean-Wheat Double Cropping in Rio Grande do Sul.* Manuscript in preparation for *International Transactions in Operational Research*.

The evaluated system is **double cropping** (sequential soybean-wheat succession), not intercropping in the strict agronomic sense (simultaneous co-cultivation). A future-work line at the end of this notebook outlines adaptation of the framework to true intercropping systems.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ahp import evaluate_matrix
from src.maut import (apply_exponential_to_matrix, evaluate_scenarios,
                       global_utility)
from src.sensitivity import (monte_carlo_simulation, oat_sensitivity,
                              summarize_monte_carlo, utility_monte_carlo)
from src.visualization import generate_all_figures

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in (TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)


## 2. Main-criteria weights

Main-criteria weights (Economic 0.40, Agronomic 0.25, Environmental 0.20, Logistical 0.15) were elicited by **direct assignment**, validated in a consensus meeting with the six-member expert panel — not derived from a pairwise comparison matrix. Hence there is no main-criteria comparison matrix in this pipeline; only the sub-criteria are derived via AHP.

In [ ]:
MAIN_WEIGHTS = {
    "Economic": 0.40,
    "Agronomic": 0.25,
    "Environmental": 0.20,
    "Logistical": 0.15,
}
CRIT_ORDER = ["Economic", "Agronomic", "Environmental", "Logistical"]
MAIN_WEIGHTS


## 3. AHP — sub-criteria local weights (principal eigenvector method)

Sub-criterion local weights are derived from the expert pairwise comparison matrices via the principal eigenvector method, then combined with the directly assigned main-criteria weights: `final_weight = local_weight x main_weight`.

Consistency is checked via the Consistency Ratio (CR); the Environmental sub-criteria block has CR = 0.0463 (within the acceptable 0.10 threshold).

In [ ]:
sub = pd.read_csv(DATA_DIR / "pairwise_subcriteria.csv")
value_cols = list(sub.columns[2:])

records = []
local_per_crit = []

for crit in CRIT_ORDER:
    block = sub[sub.parent_criterion == crit]
    cols = [c for c in value_cols if block[c].notna().all()]
    matrix = block[cols].values.astype(float)
    res = evaluate_matrix(matrix)
    local_per_crit.append(res["weights"])
    print(f"[{crit}] lambda_max={res['lambda_max']:.4f}  CR={res['CR']:.4f} "
          f"({'consistent' if res['consistent'] else 'INCONSISTENT'})")
    for name, lw in zip(block.subcriterion.tolist(), res["weights"]):
        fw = MAIN_WEIGHTS[crit] * lw
        records.append({
            "criterion": crit,
            "subcriterion": name,
            "main_weight": MAIN_WEIGHTS[crit],
            "local_weight": lw,
            "final_weight": fw,
            "CR": res["CR"],
        })

weights_df = pd.DataFrame(records)
weights_df.to_csv(TABLES_DIR / "ahp_weights.csv", index=False)
print(f"\nSum of final weights = {weights_df.final_weight.sum():.6f}")
weights_df


## 4. MAUT baseline (linear utility)

Scenario A is the soybean-wheat double cropping configuration; B and C are the comparison scenarios defined in the dissertation. Expected baseline values (for verification against the published results): **U(A) = 0.8872, U(B) = 0.5953, U(C) = 0.2421**.

In [ ]:
final_weights = weights_df["final_weight"].values
main_weights = np.array([MAIN_WEIGHTS[c] for c in CRIT_ORDER])

utilities_df = pd.read_csv(DATA_DIR / "utility_values.csv")
utility_matrix = utilities_df[["scenario_A", "scenario_B", "scenario_C"]].values

baseline = evaluate_scenarios(final_weights, utility_matrix, scenario_labels=["A", "B", "C"])
baseline.to_csv(TABLES_DIR / "maut_baseline.csv", index=False)
baseline.round(4)


## 5. MAUT with exponential utility (risk aversion)

Three levels of risk aversion are tested via the exponential utility transform, with $\rho \in \{0.5, 1.0, 2.0\}$, to check whether the ranking A > B > C is preserved under a risk-averse decision maker.

In [ ]:
exp_results = []
for rho in [0.5, 1.0, 2.0]:
    u_exp = apply_exponential_to_matrix(utility_matrix, rho=rho)
    scen = evaluate_scenarios(final_weights, u_exp, scenario_labels=["A", "B", "C"])
    scen["rho"] = rho
    exp_results.append(scen)

exp_df = pd.concat(exp_results, ignore_index=True)
exp_df.to_csv(TABLES_DIR / "maut_exponential.csv", index=False)
exp_df.pivot(index="rho", columns="scenario", values="global_utility").round(4)


## 6. One-at-a-time (OAT) sensitivity

Each main-criterion weight is perturbed by -20%, -10%, +10%, +20% (holding the others proportionally rescaled), to check how sensitive the global utility scores are to each criterion's weight.

In [ ]:
oat = oat_sensitivity(main_weights, local_per_crit, utility_matrix,
                       perturbations=[-0.20, -0.10, 0.0, 0.10, 0.20])
oat["criterion"] = oat["criterion_idx"].map(dict(enumerate(CRIT_ORDER)))
oat.to_csv(TABLES_DIR / "oat_sensitivity.csv", index=False)
oat[["criterion", "perturbation", "U_A", "U_B", "U_C"]].round(4)


## 7. Monte Carlo simulation on weights (10,000 iterations, ±20%)

Weights are jointly perturbed (seed = 42) to test the robustness of the A > B > C ranking under simultaneous, simulated uncertainty in criteria weights.

In [ ]:
mc_samples = monte_carlo_simulation(main_weights, local_per_crit, utility_matrix,
                                     n_iterations=10000, perturbation_range=0.20,
                                     random_seed=42)
mc_summary = summarize_monte_carlo(mc_samples)
mc_samples.to_csv(TABLES_DIR / "monte_carlo_samples.csv", index=False)
mc_summary.to_csv(TABLES_DIR / "monte_carlo_summary.csv", index=False)

print(f"Ranking A > B > C preserved in "
      f"{100 * mc_samples['ranking_preserved'].mean():.2f}% of "
      f"{len(mc_samples):,} iterations")
mc_summary.round(4)


## 8. Utility-value Monte Carlo (±10% and ±20%)

A second, complementary robustness check: instead of perturbing weights, the underlying utility values themselves are perturbed, at two severity levels.

In [ ]:
for pct, tag in [(0.10, "10pct"), (0.20, "20pct")]:
    umc = utility_monte_carlo(final_weights, utility_matrix,
                               n_iterations=10000, perturbation_range=pct,
                               random_seed=42)
    umc_summary = summarize_monte_carlo(umc)
    umc.to_csv(TABLES_DIR / f"utility_monte_carlo_{tag}_samples.csv", index=False)
    umc_summary.to_csv(TABLES_DIR / f"utility_monte_carlo_{tag}_summary.csv", index=False)

    print(f"+/- {int(pct*100)}%  ranking preserved: "
          f"{100 * umc['ranking_preserved'].mean():.2f}%")
    display(umc_summary.round(4))


## 9. Figures

Generates the tornado plot (OAT), the weight Monte Carlo distribution, and the exponential utility comparison — saved to `results/figures/`.

In [ ]:
generate_all_figures(mc_samples, oat, exp_df, baseline, FIGURES_DIR)
print(f"Saved to {FIGURES_DIR}/")


## 10. Summary

Four independent robustness procedures (OAT, weight Monte Carlo, utility-value Monte Carlo at ±10%/±20%, and exponential utility under three risk-aversion levels) all preserve the ranking **A > B > C**, supporting the conclusion that the soybean-wheat double cropping configuration (Scenario A) dominates the alternatives evaluated.

**Future work:** the AHP-MAUT framework presented here was built for double cropping (sequential succession). Adapting it to true intercropping systems (simultaneous co-cultivation) would require revisiting the agronomic sub-criteria set, since spatial/temporal competition effects between species are not captured by the current structure.